In [0]:
# Create a Spark session object named correlation
from pyspark.sql import SparkSession
correlation = SparkSession.builder.appName("correlation").getOrCreate()

In [0]:
# Upload website_traffic.csv to Databricks (first row = header)
df = correlation.read.csv(
    "/Workspace/Users/jessica.vargas006@mymdc.net/website_traffic.csv",
    header=True,
    inferSchema=True
)
df.show(5)

+------+---------------+
|clicks|items_purchased|
+------+---------------+
|     2|              1|
|     9|              1|
|    12|              0|
|    13|              2|
|     9|              1|
+------+---------------+
only showing top 5 rows


In [0]:
# Count the number of records 
df.count()

177

In [0]:
display(df)

clicks,items_purchased
2,1
9,1
12,0
13,2
9,1
7,1
19,4
23,7
4,0
9,2


Databricks visualization. Run in Databricks to view.

In [0]:
#Calculate the Pearson correlation coefficient and explain your thoughts about it.
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

assembler = VectorAssembler(
    inputCols=["items_purchased", "clicks"],
    outputCol="features"
)

df_vec = assembler.transform(df).select("features")

pearson_matrix = Correlation.corr(df_vec, "features", "pearson").collect()[0][0]
pearson_r = pearson_matrix[0, 1]
pearson_r

np.float64(0.11964442630209719)

If Pearson r is positive → as clicks increase, items purchased tends to increase (positive relationship).

If Pearson r is negative → as clicks increase, items purchased tends to decrease (inverse relationship).

If Pearson r is near 0 → little/no linear relationship.

In [0]:
# Calculate the Spearman correlation coefficient and explain your thoughts about it. 
spearman_matrix = Correlation.corr(df_vec, "features", "spearman").collect()[0][0]
spearman_rho = spearman_matrix[0, 1]
spearman_rho



np.float64(0.1193907524227867)

Spearman (ρ) is about whether the relationship is consistently increasing/decreasing (monotonic), even if it’s not perfectly linear.

If Spearman is strong but Pearson is weaker, that usually hints: “relationship exists, but it’s not perfectly linear.”

### 7) Pearson vs. Spearman — difference + which is more efficient here?

- **Pearson correlation** measures the **linear relationship** between two numeric variables (how well they fit a straight-line trend). It’s also the **default** method in Spark’s `corr()`. :contentReference[oaicite:0]{index=0}  
- **Spearman correlation** measures **rank (monotonic) correlation** — it looks at whether values tend to increase/decrease together based on their **order/ranks**, not necessarily in a perfect straight line. In Spark, it’s considered **fairly costly** because it requires ranking/sorting. :contentReference[oaicite:1]{index=1}  

**Which is more efficient for this task?**  
For this correlation task (`items_purchased` vs `clicks`), **Pearson is more efficient** because it’s simpler to compute and doesn’t require ranking like Spearman. :contentReference[oaicite:2]{index=2}
